# Training Notebook (Colab / Local)

Config-driven training pipeline. Core logic lives in `data_splitter.py`, `dataset.py`,
`model.py`, `pytorch_lightning.py`, and `training_utils.py` — this notebook just wires
them together against `configs/base.yaml`. Edit those files directly; local runs (no
`google.colab` import) pick up changes immediately, no push/pull needed.

**Kaggle auth** (only needed if `data/` isn't already present): tries, in order, an
existing `KAGGLE_API_TOKEN` env var, `~/.kaggle/access_token`, `~/.kaggle/kaggle.json`,
a Colab secret named `KAGGLE_API_TOKEN` (browser UI only), then an interactive prompt.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Works around a known Windows conda/pip OpenMP DLL conflict (harmless elsewhere).
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

GIT_URL = 'https://github.com/hagairavid18/beilinson.git'
GIT_BRANCH = 'main'

try:
    import google.colab  # noqa: F401
    PROJECT_ROOT = Path('/content/beilinson')
    if PROJECT_ROOT.exists():
        subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'pull'])
    else:
        subprocess.check_call(['git', 'clone', '--branch', GIT_BRANCH, GIT_URL, str(PROJECT_ROOT)])
except ImportError:
    # Not on Colab (e.g. a local kernel) - use the repo checkout we're already in.
    PROJECT_ROOT = Path.cwd()

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
(PROJECT_ROOT / 'data').mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / 'artifacts').mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Has data already:', any((PROJECT_ROOT / 'data').glob('*/*')))

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

In [ ]:
import yaml
import torch
import pandas as pd
import lightning.pytorch as pl
from torch.utils.data import DataLoader
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint

from dataset import WorkoutSequenceDataset
from model import SequenceClassifier
from pytorch_lightning import WorkoutLightningModule
from training_utils import ensure_artifacts, ensure_dataset, epoch_history, save_results

## Config

In [ ]:
with open(PROJECT_ROOT / 'configs' / 'base.yaml', 'r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

CONFIG

## Dataset

Downloads from Kaggle only if `data/` is empty (see `ensure_dataset` in `training_utils.py`
for the auth fallback chain).

In [ ]:
class_names = ensure_dataset(PROJECT_ROOT)
print(f'{len(class_names)} classes:', class_names)

## Build dataloaders

`ensure_artifacts` builds (or reuses) the clip/frame manifests and the fixed-length
`f00..fNN` frame-sequence CSV. Each split gets its own `WorkoutSequenceDataset`, wrapped
in a `DataLoader`. Manifest frame paths are stored as `data/<class>/<file>`, relative to
`PROJECT_ROOT` — so `PROJECT_ROOT` itself (not `PROJECT_ROOT / 'data'`) is the dataset's
`data_root`.

In [ ]:
pl.seed_everything(CONFIG['mode']['seed'], workers=True)

artifacts = ensure_artifacts(CONFIG, PROJECT_ROOT)

data_cfg = CONFIG['data']
image_size = data_cfg['image_size']
batch_size = data_cfg['batch_size']
num_workers = data_cfg['num_workers']
pin_memory = torch.cuda.is_available()

train_dataset = WorkoutSequenceDataset(artifacts['sequence_manifest'], PROJECT_ROOT, split='train', image_size=image_size)
val_dataset = WorkoutSequenceDataset(artifacts['sequence_manifest'], PROJECT_ROOT, split='val', image_size=image_size)
test_dataset = WorkoutSequenceDataset(artifacts['sequence_manifest'], PROJECT_ROOT, split='test', image_size=image_size)

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=num_workers, pin_memory=pin_memory, persistent_workers=bool(num_workers),
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=pin_memory, persistent_workers=bool(num_workers),
)
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=pin_memory, persistent_workers=bool(num_workers),
)

label_map = pd.read_csv(artifacts['label_map'])
num_classes = int(label_map['label_id'].nunique())

print(f'{num_classes} classes')
print(f'train {len(train_dataset)} / val {len(val_dataset)} / test {len(test_dataset)} clips')

## Build model

`SequenceClassifier` runs a small CNN (`FrameEncoder`) over every frame independently,
then pools the per-frame embeddings across time (mean/max/LSTM, per `temporal_pooling`)
before a linear classifier head. `WorkoutLightningModule` wraps it with the train/val/test/
predict steps and the optimizer.

In [ ]:
model_cfg = CONFIG['model']
training_cfg = CONFIG['training']

model = SequenceClassifier(
    num_classes=num_classes,
    in_channels=model_cfg['in_channels'],
    hidden_dims=tuple(model_cfg['hidden_dims']),
    embedding_dim=model_cfg['embedding_dim'],
    dropout=model_cfg['dropout'],
    temporal_pooling=model_cfg['temporal_pooling'],
)
lit_module = WorkoutLightningModule(
    model=model,
    lr=training_cfg['lr'],
    weight_decay=training_cfg['weight_decay'],
)

lit_module

## Build trainer

Callbacks: `ModelCheckpoint` keeps the best epoch by `monitor`/`monitor_mode`,
`EarlyStopping` stops after `patience` epochs without improvement, `LearningRateMonitor`
logs the LR each epoch. `precision: auto` picks fp16 on GPU, fp32 on CPU.

In [ ]:
checkpoint_dir = PROJECT_ROOT / training_cfg.get('checkpoint_dir', 'artifacts/checkpoints')
checkpoint_dir.mkdir(parents=True, exist_ok=True)

monitor = training_cfg.get('monitor', 'val_acc')
monitor_mode = training_cfg.get('monitor_mode', 'max')

callbacks = [
    ModelCheckpoint(
        dirpath=checkpoint_dir,
        filename='epoch{epoch:02d}-{val_acc:.3f}',
        monitor=monitor,
        mode=monitor_mode,
        save_top_k=1,
    ),
    EarlyStopping(monitor=monitor, mode=monitor_mode, patience=training_cfg.get('patience', 4)),
    LearningRateMonitor(logging_interval='epoch'),
]

precision = training_cfg.get('precision', '32-true')
if precision == 'auto':
    precision = '16-mixed' if torch.cuda.is_available() else '32-true'

trainer = pl.Trainer(
    max_epochs=training_cfg.get('max_epochs', 10),
    accelerator=training_cfg.get('accelerator', 'auto'),
    devices=training_cfg.get('devices', 'auto'),
    precision=precision,
    log_every_n_steps=training_cfg.get('log_every_n_steps', 10),
    default_root_dir=str(PROJECT_ROOT / 'artifacts'),
    callbacks=callbacks,
)

## Train

Live progress bar with per-step loss/accuracy comes from Lightning's `Trainer.fit`
directly below.

In [ ]:
trainer.fit(lit_module, train_loader, val_loader)

## Training history

Per-epoch train/val loss and accuracy, read back from the CSV logger.

In [ ]:
epoch_history(trainer)

## Evaluate

In [ ]:
test_results = trainer.test(lit_module, dataloaders=test_loader, verbose=True)

## Predict & save

In [ ]:
prediction_batches = trainer.predict(lit_module, dataloaders=test_loader)
summary = save_results(trainer, artifacts, test_results, prediction_batches, PROJECT_ROOT)
summary

In [ ]:
import json

summary_path = PROJECT_ROOT / 'artifacts' / 'training_summary.json'
with open(summary_path, 'r', encoding='utf-8') as handle:
    summary = json.load(handle)

summary